# Cell-Cell Communication Prediction using Graph Neural Networks
### COMP-4740 — Winter 2026

| | |
|---|---|
| **Task** | Link prediction on cell-cell communication networks |
| **Models** | Baseline GNN · GCN · GAT |
| **Protocol** | 90-10 train-test split, balanced negative sampling (SEGCECO) |

### Datasets
| Name | Source | Tissue | Cells | Notes |
|---|---|---|---|---|
| HumanD1–D4 | Baron et al. 2016 (GSE84133) via SEGCECO | Human pancreas | 1303–3605 | Primary datasets |
| MouseD1–D2 | Baron et al. 2016 (GSE84133) via SEGCECO | Mouse pancreas | 822–1064 | Cross-species validation |
| KangPBMC | Kang et al. 2018 (GSE96583) via LIANA+ | Human PBMC | ~2400 | Cross-tissue validation |

## 0 · Setup

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('.'))
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.utils import set_seed
set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')

PyTorch : 2.5.1
Device  : cpu


---
## Part 1 — Primary Datasets: Human Pancreas (SEGCECO)

Baron et al. 2016 human pancreas scRNA-seq data.  
Pre-processed edgelists and 300 IG-selected gene features from the SEGCECO GitHub repository.  
Graph = SoptSC-derived cell-cell communication network (individual cells as nodes).

In [2]:
from src.data_loader import load_all_human_datasets
from src.graph_builder import build_link_prediction_data
from src.trainer import split_data

print('Loading SEGCECO Human Pancreas datasets (HumanD1–D4)...')
human_raw = load_all_human_datasets(n_features=300)

human_splits = {}
for name, data in human_raw.items():
    lp = build_link_prediction_data(data)
    train, _, test = split_data(lp, test_ratio=0.1, seed=42)
    human_splits[name] = (train, test)

print('\nSplit summary:')
for name, (tr, te) in human_splits.items():
    n_train = tr.pos_edge_label_index.shape[1] + tr.neg_edge_label_index.shape[1]
    n_test  = te.pos_edge_label_index.shape[1] + te.neg_edge_label_index.shape[1]
    print(f'  {name}: train={n_train} | test={n_test}')

Loading SEGCECO Human Pancreas datasets (HumanD1–D4)...
[HumanD1] nodes=1930, edges=67882, features=300, cell-types=1
[HumanD2] nodes=1724, edges=60446, features=300, cell-types=1
[HumanD3] nodes=3597, edges=125700, features=300, cell-types=1
[HumanD4] nodes=1282, edges=45458, features=300, cell-types=1
[graph_builder] HumanD1 | 1930 nodes | 67882 positive CCN edges
[graph_builder] HumanD2 | 1724 nodes | 60446 positive CCN edges
[graph_builder] HumanD3 | 3597 nodes | 125700 positive CCN edges
[graph_builder] HumanD4 | 1282 nodes | 45458 positive CCN edges

Split summary:
  HumanD1: train=122188 | test=13576
  HumanD2: train=108804 | test=12088
  HumanD3: train=226260 | test=25140
  HumanD4: train=81826 | test=9090


In [ ]:
from src.models import BaselineGNN, GCN, GAT
from src.trainer import train_model

HIDDEN_DIM = 128
EPOCHS     = 400
LR         = 1e-3

MODEL_DEFS = {
    'Baseline GNN': (BaselineGNN, {}),
    'GCN':          (GCN,         {}),
    'GAT':          (GAT,         {'heads': 4}),
}

human_results   = {}
human_histories = {}

for ds_name, (train_data, test_data) in human_splits.items():
    in_dim = train_data.x.shape[1]
    print(f'\n{"="*60}\n  {ds_name}  |  in_dim={in_dim}\n{"="*60}')
    human_results[ds_name]   = {}
    human_histories[ds_name] = {}

    for model_name, (ModelCls, kw) in MODEL_DEFS.items():
        print(f'\n  [{model_name}]')
        set_seed(42)
        model = ModelCls(in_dim=in_dim, hidden_dim=HIDDEN_DIM, **kw)
        _, metrics, history = train_model(
            model, train_data, test_data,
            epochs=EPOCHS, lr=LR, device=DEVICE, verbose=True
        )
        human_results[ds_name][model_name]   = metrics
        human_histories[ds_name][model_name] = history

print('\nHuman pancreas training complete.')


  HumanD1  |  in_dim=300

  [Baseline GNN]
  Epoch   50 | loss=0.5023 | AUC=0.8386 | F1=0.7849
  Epoch  100 | loss=0.3490 | AUC=0.9255 | F1=0.8730


In [ ]:
from src.utils import results_table, plot_roc_curves, plot_pr_curves, plot_training_history, plot_metric_comparison

for ds_name, model_results in human_results.items():
    print(f'\n### {ds_name}')
    results_table(model_results)
    plot_roc_curves(model_results, title=ds_name)
    plot_pr_curves(model_results,  title=ds_name)

for ds_name, histories in human_histories.items():
    plot_training_history(histories, title=ds_name)

---
## Part 2 — Secondary Dataset A: Mouse Pancreas (SEGCECO)

Baron et al. 2016 **mouse** pancreas — same SoptSC pipeline as human data.  
Tests **cross-species generalization** of our GNN architectures.

In [ ]:
from src.secondary_datasets import load_all_mouse_datasets

print('Loading SEGCECO Mouse Pancreas datasets (MouseD1–D2)...')
mouse_raw = load_all_mouse_datasets(n_features=300)

mouse_splits = {}
for name, data in mouse_raw.items():
    lp = build_link_prediction_data(data)
    train, _, test = split_data(lp, test_ratio=0.1, seed=42)
    mouse_splits[name] = (train, test)

mouse_results   = {}
mouse_histories = {}

for ds_name, (train_data, test_data) in mouse_splits.items():
    in_dim = train_data.x.shape[1]
    print(f'\n{"="*60}\n  {ds_name}  |  in_dim={in_dim}\n{"="*60}')
    mouse_results[ds_name]   = {}
    mouse_histories[ds_name] = {}

    for model_name, (ModelCls, kw) in MODEL_DEFS.items():
        print(f'\n  [{model_name}]')
        set_seed(42)
        model = ModelCls(in_dim=in_dim, hidden_dim=HIDDEN_DIM, **kw)
        _, metrics, history = train_model(
            model, train_data, test_data,
            epochs=EPOCHS, lr=LR, device=DEVICE, verbose=True
        )
        mouse_results[ds_name][model_name]   = metrics
        mouse_histories[ds_name][model_name] = history

print('\nMouse pancreas training complete.')

In [ ]:
for ds_name, model_results in mouse_results.items():
    print(f'\n### {ds_name}')
    results_table(model_results)
    plot_roc_curves(model_results, title=ds_name)
    plot_pr_curves(model_results,  title=ds_name)

---
## Part 3 — Secondary Dataset B: Kang 2018 PBMC (LIANA+ Dataset)

**Kang et al. 2018** (GSE96583) — ~25k PBMCs from 8 lupus patients.  
Used as the tutorial dataset in **LIANA+** (the leading Python CCC framework).  
Cell types: CD4T, CD8T, B cells, NK, CD14 Monocytes, FCGR3A Monocytes, DCs, Megakaryocytes.

Graph construction: kNN cosine similarity graph on PCA embeddings (n=15 neighbours),  
mirroring the SoptSC cell similarity approach used in SEGCECO.  
This tests **cross-tissue generalization** (immune cells vs. pancreas).

> First run downloads ~200 MB from figshare — cached after that.

In [ ]:
from src.secondary_datasets import load_kang_pbmc

print('Loading Kang 2018 PBMC (LIANA+ dataset)...')
kang_data = load_kang_pbmc(n_cells_per_type=300, n_features=300, n_neighbors=15)

In [ ]:
kang_lp = build_link_prediction_data(kang_data)
kang_train, _, kang_test = split_data(kang_lp, test_ratio=0.1, seed=42)

n_train = kang_train.pos_edge_label_index.shape[1] + kang_train.neg_edge_label_index.shape[1]
n_test  = kang_test.pos_edge_label_index.shape[1]  + kang_test.neg_edge_label_index.shape[1]
print(f'Train edges: {n_train}')
print(f'Test  edges: {n_test}')

kang_results   = {}
kang_histories = {}
in_dim = kang_data.x.shape[1]

for model_name, (ModelCls, kw) in MODEL_DEFS.items():
    print(f'\n  [KangPBMC — {model_name}]')
    set_seed(42)
    model = ModelCls(in_dim=in_dim, hidden_dim=HIDDEN_DIM, **kw)
    _, metrics, history = train_model(
        model, kang_train, kang_test,
        epochs=EPOCHS, lr=LR, device=DEVICE, verbose=True
    )
    kang_results[model_name]   = metrics
    kang_histories[model_name] = history

print('\nKang PBMC training complete.')

In [ ]:
print('\n### KangPBMC (LIANA+ dataset)')
results_table(kang_results)
plot_roc_curves(kang_results,   title='KangPBMC')
plot_pr_curves(kang_results,    title='KangPBMC')
plot_training_history(kang_histories, title='KangPBMC')

---
## Part 4 — Cross-Dataset Comparison

Aggregate all results across all 7 datasets to assess model consistency.

In [ ]:
from src.utils import plot_cross_dataset_heatmap

# Combine all dataset results
all_results = {**human_results, **mouse_results, 'KangPBMC': kang_results}

plot_cross_dataset_heatmap(all_results, metric='auc_roc')
plot_cross_dataset_heatmap(all_results, metric='f1')
plot_cross_dataset_heatmap(all_results, metric='accuracy')

In [ ]:
metrics_keys = ['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1']

rows = []
for ds_name, model_results in all_results.items():
    for model_name, m in model_results.items():
        rows.append({
            'Dataset': ds_name,
            'Model':   model_name,
            **{k: round(m[k], 4) for k in metrics_keys}
        })

df_all = pd.DataFrame(rows)

print('=== Full Results Table (all 7 datasets × 3 models) ===')
print(df_all.to_string(index=False))

In [ ]:
print('\n=== Average across ALL datasets ===')
avg_all = df_all.groupby('Model')[metrics_keys].mean().round(4)
print(avg_all.to_string())

print('\n=== Average: Human Pancreas only ===')
avg_human = df_all[df_all.Dataset.str.startswith('Human')].groupby('Model')[metrics_keys].mean().round(4)
print(avg_human.to_string())

print('\n=== Average: Mouse Pancreas only ===')
avg_mouse = df_all[df_all.Dataset.str.startswith('Mouse')].groupby('Model')[metrics_keys].mean().round(4)
print(avg_mouse.to_string())

In [ ]:
from src.utils import plot_metric_comparison

# Overall average comparison
avg_dict = {model: {k: float(avg_all.loc[model, k]) for k in metrics_keys}
            for model in avg_all.index}
plot_metric_comparison(avg_dict, title='Average (All Datasets)')

---
## Part 5 — Save All Results

In [ ]:
os.makedirs('results', exist_ok=True)

df_all.to_csv('results/all_metrics.csv', index=False)
avg_all.to_csv('results/average_metrics_all.csv')
avg_human.to_csv('results/average_metrics_human.csv')
avg_mouse.to_csv('results/average_metrics_mouse.csv')

print('Saved:')
print('  results/all_metrics.csv')
print('  results/average_metrics_all.csv')
print('  results/average_metrics_human.csv')
print('  results/average_metrics_mouse.csv')
print('  results/*.png  (all plots)')